# Mini-Dorado Pipeline: Distilled RL for Small GPUs

This notebook implements the core ideas of the Dorado paper in a lightweight, debug-friendly pipeline:
1. **Stage 1**: Cold-start SFT on non-verifiable data.
2. **Stage 2**: Dual-reward RL (PPO) combining correctness and preference.
3. **Evaluation**: Comparison across stages.

## 🛠️ Step 0: Setup & Install Dependencies

In [1]:
# Run this if you need to install dependencies
!pip install -q torch transformers trl peft datasets accelerate bitsandbytes pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.9/530.9 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 19.7 MB/s eta 0:00:00:00:0100:01


## 🎯 Stage 1: Cold-Start SFT
We train on general instructions to establish a quality baseline.

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
import os

model_name = "Qwen/Qwen2.5-0.5B"
output_dir = "./coldstart_model"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

lora_config = LoraConfig(
    r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

# Using Alpaca dataset (Parquet format) to avoid script loading errors
dataset = load_dataset("tatsu-lab/alpaca", split="train[:100]")

def tokenize_fn(ex):
    prompt = f"Instruction: {ex['instruction']}\nInput: {ex['input']}\nResponse: {ex['output']}"
    return tokenizer(prompt, truncation=True, max_length=512)

tokenized_dataset = dataset.map(tokenize_fn, remove_columns=dataset.column_names)

args = TrainingArguments(
    output_dir=output_dir, 
    per_device_train_batch_size=2, 
    num_train_epochs=1, 
    logging_steps=5, 
    report_to="none",
    logging_dir="./logs",
    disable_tqdm=False
)

trainer = Trainer(model=model, args=args, train_dataset=tokenized_dataset, data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False))
trainer.train()
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Step,Training Loss
5,2.067644
10,1.970896
15,2.209354
20,1.786319
25,2.074321
30,1.585429
35,1.831511
40,1.698220
45,1.688494
50,1.695853


('./coldstart_model/tokenizer_config.json',
 './coldstart_model/chat_template.jinja',
 './coldstart_model/tokenizer.json')

In [13]:
# ! pip uninstall --yes trl
! pip install trl==0.11.4

  Using cached trl-0.11.4-py3-none-any.whl.metadata (12 kB)
Using cached trl-0.11.4-py3-none-any.whl (316 kB)


## 🎯 Stage 2: Dual-Reward PPO
We optimize for both **Correctness** (reasoning) and **Preference** (quality).

In [ ]:
```python
import os
import torch
from datasets import Dataset
from tqdm.auto import tqdm
from trl import PPOConfig, PPOTrainer, AutoModelForCausalLMWithValueHead, create_reference_model

if os.path.exists(output_dir):
    # Load model with value head
    model = AutoModelForCausalLMWithValueHead.from_pretrained(output_dir, torch_dtype="auto", device_map="auto")
    
    # Fix for TRL 0.12+ : ensure generation_config is attached
    if not hasattr(model, "generation_config"):
        model.generation_config = model.pretrained_model.generation_config
        
    ref_model = create_reference

Using standard PPO implementation


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/trl/experimental/ppo/modeling_value_head.py:273: FutureWarning: The `AutoModelForCausalLMWithValueHead` is now located in `trl.experimental`. Please update your imports to `from trl.experimental.ppo import AutoModelForCausalLMWithValueHead`. The current import path will be removed and no longer supported in TRL 0.29. For more information, see https://github.com/huggingface/trl/issues/4223.
  model = cls(pretrained_model, **multi_adapter_args, **trl_model_args)
<string>:144: FutureWarning: The `PPOConfig` is now located in `trl.experimental`. Please update your imports to `from trl.experimental.ppo import PPOConfig`. The current import path will be removed and no longer supported in TRL 0.29. For more information, see https://github.com/huggingface/trl/issues/4223.


Map:   0%|          | 0/2 [00:00<?, ? examples/s]

/tmp/ipython-input-4188805033.py:76: FutureWarning: The `PPOTrainer` is now located in `trl.experimental`. Please update your imports to `from trl.experimental.ppo import PPOTrainer`. The current import path will be removed and no longer supported in TRL 0.29. For more information, see https://github.com/huggingface/trl/issues/4223.
  ppo_trainer = PPOTrainer(**legacy_kwargs_attempt1)
/tmp/ipython-input-4188805033.py:79: FutureWarning: The `PPOTrainer` is now located in `trl.experimental`. Please update your imports to `from trl.experimental.ppo import PPOTrainer`. The current import path will be removed and no longer supported in TRL 0.29. For more information, see https://github.com/huggingface/trl/issues/4223.
  ppo_trainer = PPOTrainer(**legacy_kwargs_attempt2)
/tmp/ipython-input-4188805033.py:83: FutureWarning: The `PPOTrainer` is now located in `trl.experimental`. Please update your imports to `from trl.experimental.ppo import PPOTrainer`. The current import path will be removed 

RuntimeError: Couldn't construct PPOTrainer with tried signatures: PPOTrainer.__init__() missing 2 required positional arguments: 'train_dataset' and 'value_model'

## 📊 Step 3: Evaluation
Compare the Base Model, SFT, and Dorado (PPO).

In [ ]:
import pandas as pd
import os
from transformers import AutoTokenizer, AutoModelForCausalLM

# Use Instruction format to match SFT training
eval_prompts = [
    "Instruction: What is 3+4?\nInput: \nResponse:",
    "Instruction: Explain 1+1 in one concise sentence.\nInput: \nResponse:"
]
results = []

def eval_model(path, name):
    is_local = os.path.exists(path)
    if not is_local and "model" in path:
        print(f"Skipping {name}: Path not found.")
        return []
    
    print(f"Evaluating {name}...")
    try:
        m = AutoModelForCausalLM.from_pretrained(path, torch_dtype="auto", device_map="auto", local_files_only=is_local)
        t = AutoTokenizer.from_pretrained(path if is_local else model_name)
        
        model_res = []
        for p in eval_prompts:
            inputs = t(p, return_tensors="pt").to(m.device)
            # Increase tokens for SFT to see full response
            out = m.generate(**inputs, max_new_tokens=40, pad_token_id=t.eos_token_id)
            full_output = t.decode(out[0], skip_special_tokens=True)
            # Extract ONLY the response part
            ans = full_output.split("Response:")[-1].strip()
            model_res.append({"Model": name, "Prompt": p.split('\n')[0], "Output": ans})
        return model_res
    except Exception as e:
        print(f"Error evaluating {name}: {e}")
        return []

# Evaluate Base Model
results += eval_model(model_name, "Base")

# Evaluate SFT Model (Stage 1)
results += eval_model("./coldstart_model", "SFT")

# Evaluate Dorado Model (Stage 2)
results += eval_model("./dorado_toy_model", "Dorado")

if results:
    df = pd.DataFrame(results)
    print("\n--- Comparison Table ---")
    # Set options for better visibility
    pd.set_option('display.max_colwidth', None)
    print(df)
    
    # SAVE TO CSV
    df.to_csv("evaluation_results.csv", index=False)
    print("\n✅ Results saved to 'evaluation_results.csv'")
else:
    print("No results to display.")

: 